# Run and inspect the pattern search

Level: 2. Prerequisites: Level 1 of this topic and levels 1-2 of comminution. Reading time: 30 minutes.

## Objectives

Run OreFlow's production optimizer on the soft porphyry case, inspect its actual evaluation trace and test whether the final point satisfies each declared physical constraint.

## Setup

```bash
pip install numpy==2.2.6 scipy==1.17.1 pandas==2.2.3 matplotlib==3.11.2 nbclient==0.11.0 nbformat==5.11.1 ipykernel==7.4.0 SALib==1.6.0
```

The OreFlow pipeline is imported from `data-pipeline/`; this notebook never installs packages.

Configure imports and record versions. This makes the method run reproducible.

In [1]:
# Import analysis tools and record their exact versions.
import importlib.metadata
import json
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Resolve the local pipeline from the repository-relative layout.
repo = Path('../../../').resolve()
sys.path.insert(0, str(repo / 'data-pipeline'))
for package in ('numpy', 'scipy', 'pandas', 'matplotlib', 'nbclient', 'nbformat', 'ipykernel'):
    print(package, importlib.metadata.version(package))

numpy 2.2.6
scipy 1.17.1
pandas 2.2.3
matplotlib 3.11.2
nbclient 0.11.0
nbformat 5.11.1
ipykernel 7.4.0


## Data

Owner: OreFlow authors; the soft-porphyry case and Contract 1 are versioned at `data/derived/cases/copper_porphyry_soft.json` and `data/derived/contract/operating_contract.json`, MIT project licence. Repository: https://github.com/fsantibanezleal/CAOS_OreFlow.

## Method

The generalized pattern search tests mesh neighbours in bounded decision coordinates and applies a progressive barrier to grade, power and water violations. This notebook calls `optimize` from `pipeline/methods/optimization.py`; it does not reproduce the optimizer. The run disables the optional surrogate screen to isolate the engine-evaluated poll.

Load the production case, contract and optimizer; run one deterministic metal-recovery search without a weight path.

In [2]:
from pipeline.cases.catalog import CASES
from pipeline.cases.catalog import CASES
from pipeline.methods.optimization import optimize

# Re-run the product optimizer with its declared objective and no surrogate screen.
case = next(item for item in CASES if item.id == 'copper_porphyry_soft')
contract = json.loads((repo / 'data/derived/contract/operating_contract.json').read_text(encoding='utf-8'))
search = optimize(case, case.nominal, contract, weight=1.0, path=False, screen=None)
print('status:', search['status'], '| engine evaluations:', search['evaluations'])
print('screened:', search['screened'], '| decision variables:', search['decisions'])
print('feasible optimum:', search['optimum']['feasible'] if search['optimum'] else False)

status: optimal | engine evaluations: 266
screened: False | decision variables: ['target_p80_um', 'collector_gpt', 'jg_cm_s']
feasible optimum: True


Inspect the recorded pattern-search iterations and identify the mesh stop condition. A finite evaluation budget may stop before mesh convergence.

In [3]:
# Inspect the recorded pattern-search iterations and identify the mesh stop condition.
print('start count:', len(search['starts']))
display(pd.DataFrame([{key: start.get(key) for key in ('start', 'feasible', 'evaluations', 'stop')} for start in search['starts']]))
print('converged:', search.get('converged'), '| feasible:', search['optimum']['feasible'])

start count: 6


,start,feasible,evaluations,stop
0,"{'target_p80_um': 150.0, 'collector_gpt': 25.0...",None,158,mesh
1,"{'target_p80_um': 131.25, 'collector_gpt': 18....",None,96,mesh
2,"{'target_p80_um': 243.75, 'collector_gpt': 56....",None,4,mesh
3,"{'target_p80_um': 131.25, 'collector_gpt': 56....",None,0,mesh
4,"{'target_p80_um': 243.75, 'collector_gpt': 18....",None,2,mesh
5,"{'target_p80_um': 187.5, 'collector_gpt': 37.5...",None,6,mesh


converged: True | feasible: True


Check the final slack for grade, installed mill power and water, then compare the base and optimum in process units.

In [4]:
# Check the final slack for grade, installed mill power and water, then compare the base and optimum in process units.
optimum = search['optimum']
print('constraint slacks:', optimum['slacks'])
comparison = pd.DataFrame([
    {'state': 'base', 'recovered primary (t/h)': search['base']['recovered_tph'], **search['base']['values']},
    {'state': 'optimum', 'recovered primary (t/h)': optimum['recovered_tph'], **optimum['values']},
]).set_index('state')
display(comparison)

constraint slacks: {'grade': 1.0715590708919507, 'power': 0.6730429101571644, 'water': 0.07950103059474811}


,recovered primary (t/h),grade,required_power_kw,water_m3_t,energy_kwh_t
state,,,,,
base,4.913088,26.415673,6456.494529,2.174074,10.394629
optimum,5.195311,25.071559,7269.326957,2.200499,11.868042


## Results

The live run exercises the same production function as the bake. Slacks show how far the result sits from each limit. If the run reports a budget stop, the answer remains the best feasible point found, not a proof of stationarity or global optimality.

## Summary

You executed the optimizer and checked its search trace and constraint slacks. Reproducibility of the search path does not turn a scenario result into a plant optimum.

## References

- https://doi.org/10.1137/S1052623493250780
- https://doi.org/10.1137/070692662